In [9]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from modules.datasets import DATASETS, DATA_DIR


def extract_meta(path: Path) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"file_path": str(f), "label": f.parent.name, "id": f.stem}
            for f in path.rglob("*.mp4")
        ]
    )


train_df = pd.concat(
    [extract_meta(dataset) for dataset in DATASETS["TRAIN"]],
    ignore_index=True,
)

# 2. Gather the test set
test_df = extract_meta(DATASETS["TEST"])

# Quick verification
print(f"Total Training Samples: {len(train_df)}")
print(f"Total Testing Samples: {len(test_df)}")

# Save the train and test dataframes to CSV files
df_dir = DATA_DIR / "cache" / "dataframes"
df_dir.mkdir(parents=True, exist_ok=True)
train_df.to_csv(df_dir / "train.csv", index=False)
test_df.to_csv(df_dir / "test.csv", index=False)


Total Training Samples: 30867
Total Testing Samples: 33600


In [ ]:
from matplotlib.lines import Line2D

metrics_dir = DATA_DIR / "cache" / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

train_label_distribution = train_df.groupby("label").size().reset_index(name="count")
test_label_distribution = test_df.groupby("label").size().reset_index(name="count")


mode_count = train_label_distribution["count"].mode()
std_count = train_label_distribution["count"].std()
variance_count = train_label_distribution["count"].var()


print(f"Mode: {mode_count[0]}")
print(f"Standard Deviation: {std_count}")
print(f"Variance: {variance_count}")

plt.figure(figsize=(14, 6))
counts = train_label_distribution["count"].values
labels = train_label_distribution["label"].values
x = range(len(labels))
mean_count = counts.mean()

# Line connecting points
plt.plot(x, counts, color="gray", linestyle="-", linewidth=1, alpha=0.7)

# Scatter colored by above/below/equal to mean
colors = [
    "green" if c > mean_count else ("yellow" if c < mean_count else "orange")
    for c in counts
]
plt.scatter(x, counts, c=colors, s=50, edgecolors='k', label="Sample Count")

# Mean line
plt.axhline(y=mean_count, color="r", linestyle="--", label=f"Mean: {mean_count:.2f}")

plt.title("Distribution of Samples per Class")
plt.xlabel("Class Label")
plt.ylabel("Number of Samples")
plt.xticks(x, labels, rotation=90)

# Custom legend
legend_elements = [
    Line2D([0], [0], color='r', lw=2, linestyle='--', label=f'Mean: {mean_count:.2f}'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='green', markersize=8, label='Above Mean'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='yellow', markersize=8, label='Below Mean'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='orange', markersize=8, label='Equal to Mean'),
]
plt.legend(handles=legend_elements, loc='upper right')
plt.tight_layout()
plt.show()

plt.savefig
